# Southern Strikers — PlayHQ Scraper

Pulls fixture + full scorecards for every configured team from PlayHQ's public GraphQL
endpoints (no API key needed) and writes CSVs ready for Power BI.

Run the cells top to bottom. You can safely re-run — nothing is destructive except the
final "Write CSVs" cell, which overwrites the files each time.

## 1. Imports and config

Add or update entries in `TEAMS` to cover new seasons. `team_id` is the last segment of the team page URL on PlayHQ.

In [ ]:
import json
import time
from pathlib import Path
from typing import Any

import requests
import pandas as pd

TEAMS = [
    {
        "label": "PASA Multicultural T20 Winter 2026",
        "team_id": "33d20e37",
        "url": "https://www.playhq.com/cricket-australia/org/pashtun-association-of-sa/pasa-multicultural-cricket-wintert20-winter-2026/teams/southern-strikers/33d20e37",
    },
    {
        "label": "SACA Autumn Super Cricket T20 Winter 2026",
        "team_id": "57c57ec5",
        "url": "https://www.playhq.com/cricket-australia/org/saca-super-cricket/saca-autumn-super-cricket-t20-winter-2026/teams/southern-strikers/57c57ec5",
    },
]

OUT_DIR = Path("./southern-strikers-data")   # change if you want CSVs somewhere else
OUT_DIR.mkdir(parents=True, exist_ok=True)

DISCOVER_ENDPOINT = "https://api.playhq.com/graphql"
SPECTATOR_ENDPOINT = "https://spectator.playhq.com/graphql"
DISCOVER_TENANT = "cricket-australia"
SPECTATOR_TENANT = "ca"

SLEEP_BETWEEN_GAMES = 0.4   # seconds; keep this polite

print(f"Configured teams: {len(TEAMS)}")
print(f"Output folder:    {OUT_DIR.resolve()}")

## 2. GraphQL queries

These are extracted straight from PlayHQ's own front-end bundles. Do not edit unless PlayHQ changes their schema.

In [ ]:
TEAM_FIXTURE_QUERY = """
query teamFixture($teamID: ID!) {
  discoverTeam(teamID: $teamID) {
    id name
    season { id name } grade { id name } organisation { id name }
  }
  discoverTeamFixture(teamID: $teamID) {
    id name
    grade {
      id name type
      season { id name competition { id name organisation { id name } type } }
    }
    fixture {
      games {
        id
        away { ... on ProvisionalTeam { name __typename } ... on DiscoverTeam { id name __typename } }
        home { ... on ProvisionalTeam { name __typename } ... on DiscoverTeam { id name __typename } }
        result {
          winner { name value }
          outcome { name value }
          home {
            statistics { count type { value } }
            periods { period { value } type closureStatus statistics { count type { value } } }
          }
          away {
            statistics { count type { value } }
            periods { period { value } type closureStatus statistics { count type { value } } }
          }
        }
        status { name value }
        date
        allocation { time court { name venue { name suburb } } }
        gameType { name value }
      }
    }
  }
}
"""

GAME_VIEW_QUERY = """
query gameViewSpectator($id: ID!) {
  game(id: $id) {
    id status updatedAt lastEventRecordedAt
    statistics {
      home {
        coinTossWinningResult { preference }
        statisticsV2 { type { value } count }
        players {
          id profileID name playerNumber lineupOrder permitType
          periodStatistics {
            period { value } side type
            statistics { type { value } count }
            status displayOrder
          }
          statistics { type { value } count }
        }
      }
      away {
        coinTossWinningResult { preference }
        statisticsV2 { type { value } count }
        players {
          id profileID name playerNumber lineupOrder permitType
          periodStatistics {
            period { value } side type
            statistics { type { value } count }
            status displayOrder
          }
          statistics { type { value } count }
        }
      }
      shared {
        period { value }
        players { playerID teamID role }
        dismissalType side status type
      }
    }
  }
}
"""
print("Queries loaded.")

TEAM_LADDER_QUERY = """
query teamLadder($teamID: ID!) {
  discoverTeam(teamID: $teamID) {
    id name
    grade {
      id name ladderType
      ladder(filter: {teamID: $teamID}) {
        pool { id name }
        standings {
          team { id name }
          played won lost drawn byes
          pointsFor pointsAgainst pointsDifference percentage netRunRate
          competitionPoints noResults ties
          runsFor oversFaced wicketsLost
          runsAgainst oversBowled wicketsTaken
          forfeits
        }
        gameTypeValue
      }
    }
  }
}
"""


## 3. HTTP helpers

In [ ]:
BROWSER_UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36"
)


def post_graphql(endpoint, tenant_key, tenant_val, operation, query, variables):
    r = requests.post(
        endpoint,
        headers={
            "content-type": "application/json",
            tenant_key: tenant_val,
            "accept": "*/*",
            "user-agent": BROWSER_UA,
            "origin": "https://www.playhq.com",
            "referer": "https://www.playhq.com/",
        },
        data=json.dumps({"operationName": operation, "variables": variables, "query": query}),
        timeout=30,
    )
    r.raise_for_status()
    j = r.json()
    if j.get("errors"):
        raise RuntimeError(f"GraphQL errors from {operation}: {j['errors']}")
    return j["data"]


def fetch_team_fixture(team_id):
    return post_graphql(DISCOVER_ENDPOINT, "tenant", DISCOVER_TENANT,
                        "teamFixture", TEAM_FIXTURE_QUERY, {"teamID": team_id})


def fetch_game_view(game_id):
    return post_graphql(SPECTATOR_ENDPOINT, "x-phq-tenant", SPECTATOR_TENANT,
                        "gameViewSpectator", GAME_VIEW_QUERY, {"id": game_id})


def stat_get(stats, key):
    for s in stats or []:
        t = (s or {}).get("type") or {}
        if t.get("value") == key:
            return s.get("count")
    return None


def score_from_periods(periods):
    total_score = total_wickets = total_overs = None
    for p in periods or []:
        p_stats = p.get("statistics") or []
        for key, target in (("TOTAL_SCORE", "total_score"),
                            ("TOTAL_OUTS", "total_wickets"),
                            ("TOTAL_OVERS", "total_overs")):
            v = stat_get(p_stats, key)
            if v is None:
                continue
            if target == "total_score":  total_score  = (total_score  or 0) + v
            if target == "total_wickets":total_wickets= (total_wickets or 0) + v
            if target == "total_overs":  total_overs  = (total_overs  or 0) + v
    return {"total_score": total_score, "total_wickets": total_wickets, "total_overs": total_overs}

def fetch_team_ladder(team_id):
    return post_graphql(DISCOVER_ENDPOINT, "tenant", DISCOVER_TENANT,
                        "teamLadder", TEAM_LADDER_QUERY, {"teamID": team_id})


## 4. Fetch fixtures

One request per team. Fast.

In [ ]:
fixtures = {}
for t in TEAMS:
    print(f"Fetching fixture for {t['label']}...")
    fixtures[t['team_id']] = fetch_team_fixture(t['team_id'])
    rounds = fixtures[t['team_id']]['discoverTeamFixture']
    games  = [g for r in rounds for g in r['fixture']['games']]
    finals = [g for g in games if g['status']['value'] == 'FINAL']
    print(f"  {len(games)} scheduled, {len(finals)} FINAL")

## 4b. Fetch ladders

One request per team season. Used to figure out which opponents finished above Southern Strikers.

In [ ]:
ladders = {}
for t in TEAMS:
    print(f"Fetching ladder for {t['label']}...")
    ladders[t['team_id']] = fetch_team_ladder(t['team_id'])
    standings = ladders[t['team_id']]["discoverTeam"]["grade"]["ladder"][0]["standings"]
    ss_pos = next((i+1 for i, s in enumerate(standings) if s["team"]["name"] == "Southern Strikers"), None)
    print(f"  {len(standings)} teams; Southern Strikers position: {ss_pos}")

## 5. Extract match-level rows

One row per game. Includes team scores, W/L, venue, opponent, etc.

In [ ]:
def extract_matches(team_meta, fixture_data):
    rows = []
    team = fixture_data['discoverTeam']
    for round_ in fixture_data['discoverTeamFixture']:
        grade = round_.get('grade') or {}
        for game in round_['fixture']['games']:
            result = game.get('result') or {}
            home_score = score_from_periods(((result.get('home') or {}).get('periods')) or [])
            away_score = score_from_periods(((result.get('away') or {}).get('periods')) or [])
            allocation = game.get('allocation') or {}
            court = allocation.get('court') or {}
            venue = court.get('venue') or {}
            home = game.get('home') or {}
            away = game.get('away') or {}
            winner = result.get('winner') or {}
            outcome = result.get('outcome') or {}
            our_team_id = team['id']
            our_side = 'HOME' if home.get('id') == our_team_id else ('AWAY' if away.get('id') == our_team_id else None)
            wv = winner.get('value')
            if wv is None or our_side is None:
                our_result = None
            elif wv == our_side:
                our_result = 'WIN'
            elif wv in ('HOME','AWAY'):
                our_result = 'LOSS'
            else:
                our_result = wv
            rows.append({
                'competition': team_meta['label'],
                'our_team_id': our_team_id,
                'our_team_name': team['name'],
                'our_side': our_side,
                'our_result': our_result,
                'grade_name': grade.get('name'),
                'grade_type': grade.get('type'),
                'round_name': round_.get('name'),
                'game_id': game['id'],
                'date': game.get('date'),
                'time': allocation.get('time'),
                'venue_name': venue.get('name'),
                'venue_suburb': venue.get('suburb'),
                'court_name': court.get('name'),
                'game_type': (game.get('gameType') or {}).get('name'),
                'status': (game.get('status') or {}).get('value'),
                'home_team_id': home.get('id'),
                'home_team_name': home.get('name'),
                'away_team_id': away.get('id'),
                'away_team_name': away.get('name'),
                'home_score': home_score['total_score'],
                'home_wickets': home_score['total_wickets'],
                'home_overs': home_score['total_overs'],
                'away_score': away_score['total_score'],
                'away_wickets': away_score['total_wickets'],
                'away_overs': away_score['total_overs'],
                'winner_side': winner.get('value'),
                'outcome': outcome.get('name'),
            })
    return rows

all_matches = []
all_ladder = []
for t in TEAMS:
    matches = extract_matches(t, fixtures[t['team_id']])
    standings = ladders[t['team_id']]["discoverTeam"]["grade"]["ladder"][0]["standings"]
    position_by_team = {s["team"]["name"]: i + 1 for i, s in enumerate(standings)}
    our_position = position_by_team.get("Southern Strikers")
    for m in matches:
        opp = m["home_team_name"] if m["our_side"] == "AWAY" else m["away_team_name"]
        opp_pos = position_by_team.get(opp)
        m["our_position"] = our_position
        m["opponent_position"] = opp_pos
        m["vs_top_team"] = (opp_pos is not None and our_position is not None and opp_pos < our_position)
    all_matches.extend(matches)
    for i, s in enumerate(standings, start=1):
        tm = s["team"]
        all_ladder.append({
            "competition": t["label"], "position": i,
            "team_id": tm["id"], "team_name": tm["name"],
            "played": s["played"], "won": s["won"], "lost": s["lost"],
            "drawn": s.get("drawn"), "ties": s.get("ties"), "no_results": s.get("noResults"),
            "byes": s.get("byes"), "forfeits": s.get("forfeits"),
            "competition_points": s["competitionPoints"],
            "points_for": s.get("pointsFor"), "points_against": s.get("pointsAgainst"),
            "points_difference": s.get("pointsDifference"),
            "percentage": s.get("percentage"), "net_run_rate": s["netRunRate"],
            "runs_for": s.get("runsFor"), "overs_faced": s.get("oversFaced"),
            "wickets_lost": s.get("wicketsLost"), "runs_against": s.get("runsAgainst"),
            "overs_bowled": s.get("oversBowled"), "wickets_taken": s.get("wicketsTaken"),
        })

matches_df = pd.DataFrame(all_matches)
print(f"{len(matches_df)} matches loaded")
matches_df.head(10)

## 6. Fetch every played game

One request per FINAL game. Takes ~10-30 seconds for a full season.

In [ ]:
games_data = {}
final_ids = matches_df.loc[matches_df['status'] == 'FINAL', 'game_id'].tolist()
print(f"Fetching {len(final_ids)} scorecards...")

for i, gid in enumerate(final_ids, 1):
    try:
        games_data[gid] = fetch_game_view(gid)
        print(f"  [{i:>2}/{len(final_ids)}] {gid} ok")
    except Exception as e:
        print(f"  [{i:>2}/{len(final_ids)}] {gid} FAILED: {e}")
    time.sleep(SLEEP_BETWEEN_GAMES)

print(f"\nDone. {len(games_data)} scorecards in memory.")

## 7. Extract player-level rows

Splits each player's `periodStatistics` into batting and bowling rows.

In [ ]:
def extract_player_lines(game_id, comp, date, round_name, team_side, team_name, players):
    bat, bowl = [], []
    for p in players:
        for period in p.get('periodStatistics') or []:
            stats = period.get('statistics') or []
            side = period.get('side')
            inn = (period.get('period') or {}).get('value')
            if side == team_side:
                runs = stat_get(stats, 'TOTAL_RUNS')
                balls = stat_get(stats, 'BALLS_FACED')
                fours = stat_get(stats, 'FOURS')
                sixes = stat_get(stats, 'SIXES')
                sr = stat_get(stats, 'STRIKE_RATE')
                cr = stat_get(stats, 'CURRENT_RUNS')
                cb = stat_get(stats, 'CURRENT_BALLS_FACED')
                if any(v is not None for v in (runs, balls, fours, sixes, cr)):
                    bat.append({
                        'game_id': game_id, 'competition': comp, 'date': date,
                        'round_name': round_name, 'team': team_name, 'innings': inn,
                        'player_id': p['id'], 'profile_id': p.get('profileID'),
                        'player_name': p['name'], 'lineup_order': p.get('lineupOrder'),
                        'batting_position': period.get('displayOrder'), 'status': period.get('status'),
                        'runs': runs if runs is not None else cr,
                        'balls': balls if balls is not None else cb,
                        'fours': fours, 'sixes': sixes, 'strike_rate': sr,
                    })
            else:
                overs = stat_get(stats, 'OVERS')
                runs_c = stat_get(stats, 'RUNS')
                wkts = stat_get(stats, 'WICKETS')
                econ = stat_get(stats, 'ECONOMY')
                wides = stat_get(stats, 'WIDES')
                nb = stat_get(stats, 'NO_BALLS')
                md_ = stat_get(stats, 'MAIDENS')
                if any(v is not None for v in (overs, runs_c, wkts)):
                    bowl.append({
                        'game_id': game_id, 'competition': comp, 'date': date,
                        'round_name': round_name, 'team': team_name, 'innings': inn,
                        'player_id': p['id'], 'profile_id': p.get('profileID'),
                        'player_name': p['name'],
                        'overs': overs, 'maidens': md_, 'runs_conceded': runs_c,
                        'wickets': wkts, 'economy': econ, 'wides': wides, 'no_balls': nb,
                    })
    return bat, bowl


def extract_dismissals(game_id, comp, date, round_name, shared):
    rows = []
    for s in shared or []:
        players = s.get('players') or []
        batter = next((p for p in players if p.get('role') == 'BATTING'), None)
        bowler = next((p for p in players if p.get('role') == 'BOWLING'), None)
        fielder = next((p for p in players if p.get('role') == 'FIELDING'), None)
        if not batter: continue
        rows.append({
            'game_id': game_id, 'competition': comp, 'date': date, 'round_name': round_name,
            'innings': (s.get('period') or {}).get('value'),
            'batter_player_id': batter.get('playerID'), 'batter_team_id': batter.get('teamID'),
            'dismissal_type': s.get('dismissalType'),
            'bowler_player_id': (bowler or {}).get('playerID'),
            'fielder_player_id': (fielder or {}).get('playerID'),
        })
    return rows


all_batting, all_bowling, all_dismissals = [], [], []
roster = {}

for _, m in matches_df.iterrows():
    if m['status'] != 'FINAL' or m['game_id'] not in games_data:
        continue
    game = games_data[m['game_id']]['game']
    stats = game['statistics']
    for side_key, side_val, team_name in [
        ('home', 'HOME', m['home_team_name']),
        ('away', 'AWAY', m['away_team_name']),
    ]:
        players = stats[side_key].get('players') or []
        for p in players:
            pid = p['id']
            if pid not in roster:
                roster[pid] = {'player_id': pid, 'profile_id': p.get('profileID'),
                               'player_name': p.get('name'), 'teams': set()}
            if team_name:
                roster[pid]['teams'].add(team_name)
        bat, bowl = extract_player_lines(m['game_id'], m['competition'], m['date'], m['round_name'],
                                          side_val, team_name, players)
        all_batting.extend(bat)
        all_bowling.extend(bowl)
    all_dismissals.extend(extract_dismissals(m['game_id'], m['competition'], m['date'], m['round_name'],
                                              stats.get('shared')))

batting_df = pd.DataFrame(all_batting)
bowling_df = pd.DataFrame(all_bowling)
dismissals_df = pd.DataFrame(all_dismissals)
players_df = pd.DataFrame([
    {'player_id': p['player_id'], 'profile_id': p['profile_id'], 'player_name': p['player_name'],
     'teams': ' | '.join(sorted(t for t in p['teams'] if t))}
    for p in roster.values()
])

print(f"batting_lines:   {len(batting_df):>4}")
print(f"bowling_lines:   {len(bowling_df):>4}")
print(f"dismissals:      {len(dismissals_df):>4}")
print(f"unique players:  {len(players_df):>4}")

ladder_df = pd.DataFrame(all_ladder)
print(f"ladder rows:     {len(ladder_df):>4}")

## 8. Preview each table

Sanity-check what's about to be written.

In [ ]:
print("=== Southern Strikers batting leaderboard ===")
ss_bat = batting_df[batting_df['team'] == 'Southern Strikers']
lb = (ss_bat.groupby('player_name')
        .agg(runs=('runs','sum'), innings=('runs','count'),
             fours=('fours','sum'), sixes=('sixes','sum'),
             hs=('runs','max'))
        .sort_values('runs', ascending=False).head(10))
lb

In [ ]:
print("=== Southern Strikers bowling leaderboard ===")
ss_bowl = bowling_df[bowling_df['team'] == 'Southern Strikers']
bl = (ss_bowl.groupby('player_name')
        .agg(wickets=('wickets','sum'), overs=('overs','sum'),
             runs_conceded=('runs_conceded','sum'),
             best=('wickets','max'))
        .assign(economy=lambda d: d['runs_conceded']/d['overs'].replace(0, pd.NA))
        .sort_values('wickets', ascending=False).head(10))
bl

## 9. Write CSVs for Power BI

In [ ]:
# --- Merge captain info from captains.csv (survives every scraper run) ---
captains_path = OUT_DIR / "captains.csv"

if captains_path.exists():
    captains_df = pd.read_csv(captains_path, dtype=str).fillna("")
    matches_df = matches_df.merge(
        captains_df[["game_id", "captain_player_id", "captain_player_name"]],
        on="game_id",
        how="left",
    )
    matches_df = matches_df.rename(columns={"captain_player_name": "captain"})
    print(f"merged {len(captains_df)} captain rows from {captains_path}")
else:
    # Seed a template so the user can start filling captains in
    template = matches_df[matches_df["status"] == "FINAL"][["game_id"]].copy()
    template["captain_player_id"] = ""
    template["captain_player_name"] = ""
    template.to_csv(captains_path, index=False)
    matches_df["captain_player_id"] = None
    matches_df["captain"] = None
    print(f"seeded template {captains_path} — fill it in and re-run this cell")

matches_df.to_csv(OUT_DIR / 'matches.csv', index=False)
batting_df.to_csv(OUT_DIR / 'batting_lines.csv', index=False)
bowling_df.to_csv(OUT_DIR / 'bowling_lines.csv', index=False)
dismissals_df.to_csv(OUT_DIR / 'dismissals.csv', index=False)
players_df.to_csv(OUT_DIR / 'players.csv', index=False)
ladder_df.to_csv(OUT_DIR / 'ladder.csv', index=False)

for name in ['matches.csv', 'batting_lines.csv', 'bowling_lines.csv', 'dismissals.csv', 'players.csv', 'ladder.csv']:
    p = OUT_DIR / name
    print(f"  {p.stat().st_size:>7} bytes  {p}")

print(f"\nAll CSVs written to {OUT_DIR.resolve()}")
print("Now open Power BI Desktop -> Get Data -> Folder -> point at that path.")